In [2]:
#====== Basic DS packages
import pandas as pd
import numpy as np
#===== FOR POSTGRES READ
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine
#==== ML packages
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
#======================
#READ Postgres 
load_dotenv()

database_url = os.getenv("DATABASE_URL")

if not database_url:
    raise RuntimeError("DATABASE_URL is not configured.")

engine = create_engine(database_url)

df = pd.read_sql("SELECT * FROM merged23_25", engine)

print(df.head(5))

   id         album_name     artist_name  score  \
0   1              Lahai          Sampha    8.5   
1   2  melodic nostalgia       tomcbumpz    5.0   
2   3       OK ORCHESTRA             AJR    1.0   
3   4           HELLMODE  Jeff Rosentock    8.0   
4   5     Dorothy's Harp   Dorothy Ashby    9.5   

                                               notes  listen_year artwork_url  \
0  A smooth R&B infused hip hop album. Opener is ...         2023        None   
1  This is a lowfi beats to study to ep. I didn't...         2023        None   
2  I decided I need to suffer I guess. AJR is the...         2023        None   
3  The dying masterpiece of California garage pun...         2023        None   
4  A funky bass and drum beat lay the ground for ...         2023        None   

  musicbrainz_release_id artwork_status  
0                   None        pending  
1                   None        pending  
2                   None        pending  
3                   None        pendin

In [41]:
print(df.dtypes)

id               int64
album_name         str
artist_name        str
score          float64
notes              str
listen_year      int64
dtype: object


In [42]:
def check_data(df):

    missing_count = 0

    for col in df.columns:

        if pd.api.types.is_numeric_dtype(df[col]):

            missing_count += df[col].isna().sum()
            df[col] = df[col].fillna(round(df[col].mean()))

        elif pd.api.types.is_string_dtype(df[col]):

            missing_count += df[col].isna().sum()
            df[col] = df[col].fillna("unprovided/unknown")

        else:
            raise ValueError(
                f"Column type not recognized: {df[col].dtype}"
            )
    print("Table is in Expected Format")
    print(f"Table Shape: {df.shape}")
    print(f"Number of Missing Values Filled: {missing_count}")
    return(df)

def process_string(df, col_name):
    ''' 
    A Function to handle string columns specifically for ML
    '''
    if not pd.api.types.is_string_dtype(df[col_name]):
        print("Non-string column detected. No alterations made.")
        return df

    new_name = "clean_" + col_name

    df[new_name] = (
        df[col_name]
        .str.replace('\n', ' ', regex=False)
        .str.lower()
        .str.strip()
    )

    print(f"{col_name} column regularized")
    return df

def CLEAN(df):
    df=check_data(df)
    print("================")
    for column in df.columns:
        df=process_string(df, column)
        print("================")
    print("Checks are done")
    return(df)

clean_df=CLEAN(df)
clean_df["rev_text"] = (
    clean_df["clean_album_name"] + " "
    + clean_df["clean_artist_name"] + " "
    + clean_df["clean_notes"]
)
newcols=['id', 'score','listen_year',
      'rev_text']
clean_df=clean_df[newcols]
clean_df.head(5)

Table is in Expected Format
Table Shape: (744, 6)
Number of Missing Values Filled: 4
Non-string column detected. No alterations made.
album_name column regularized
artist_name column regularized
Non-string column detected. No alterations made.
notes column regularized
Non-string column detected. No alterations made.
Checks are done


,id,score,listen_year,rev_text
0,1,8.5,2023,lahai sampha a smooth r&b infused hip hop albu...
1,2,5.0,2023,melodic nostalgia tomcbumpz this is a lowfi be...
2,3,1.0,2023,ok orchestra ajr i decided i need to suffer i ...
3,4,8.0,2023,hellmode jeff rosentock the dying masterpiece ...
4,5,9.5,2023,dorothy's harp dorothy ashby a funky bass and ...


In [43]:
X = clean_df[["rev_text", "listen_year"]]
y = clean_df["score"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)
#========================= Different model comparisons
year_model = Pipeline([
    ("scaler", StandardScaler()),
    ("regressor", Ridge(alpha=10))
])
text_model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=10_000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=1.0
        )
    ),
    ("regressor", Ridge(alpha=10))
])

preprocessor = ColumnTransformer([
    (
        "text",
        TfidfVectorizer(
            max_features=20_000,
            ngram_range=(1, 2),
            min_df=1,
            max_df=1.0
        ),
        "rev_text"
    ),
    (
        "year",
        StandardScaler(),
        ["listen_year"]
    )
])

combined_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", Ridge(alpha=10))
])

In [44]:
#MODEL FITTING
year_model.fit(
    X_train[["listen_year"]],
    y_train
)

text_model.fit(
    X_train["rev_text"],
    y_train
)

combined_model.fit(
    X_train,
    y_train
)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](2,)","['rev_text','listen_year']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,2
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('text', ...), ('year', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the tr

In [45]:
#-==========================
year_predictions = year_model.predict(
    X_test[["listen_year"]]
)

text_predictions = text_model.predict(
    X_test["rev_text"]
)

combined_predictions = combined_model.predict(
    X_test)

In [46]:
def evaluate_model(name, predictions, y_test):
    rounded = (predictions * 2).round() / 2

    mae = mean_absolute_error(y_test, predictions)
    rmse = mean_squared_error(y_test, predictions) ** 0.5
    r2 = r2_score(y_test, predictions)

    exact = (rounded == y_test).mean()
    within_half = (abs(rounded - y_test) <= 0.5).mean()

    print(f"\n{name}")
    print("-" * len(name))
    print(f"MAE:               {mae:.3f}")
    print(f"RMSE:              {rmse:.3f}")
    print(f"R²:                {r2:.3f}")
    print(f"Exact rating:      {exact:.2%}")
    print(f"Within 0.5 points: {within_half:.2%}")


evaluate_model("Year only", year_predictions, y_test)
evaluate_model("Text only", text_predictions, y_test)
evaluate_model("Text + Year", combined_predictions, y_test)




Year only
---------
MAE:               1.170
RMSE:              1.550
R²:                -0.001
Exact rating:      16.11%
Within 0.5 points: 38.26%

Text only
---------
MAE:               1.064
RMSE:              1.428
R²:                0.150
Exact rating:      16.11%
Within 0.5 points: 42.95%

Text + Year
-----------
MAE:               1.079
RMSE:              1.443
R²:                0.133
Exact rating:      15.44%
Within 0.5 points: 42.28%


In [47]:
#Making score bins for stratified cross-validation
print(clean_df["score"].value_counts().sort_index())

score_bins = pd.cut(
    clean_df["score"],
    bins=[-0.01, 5, 6.5, 7.5, 8.5, 10],
    labels=[
        "low",
        "mid_low",
        "mid",
        "high",
        "very_high"
    ]
)

print(score_bins.value_counts().sort_index())

score
0.0       1
1.0       3
1.5       1
2.0       6
2.5       1
3.0       7
3.5       4
4.0      10
4.5      11
5.0      10
5.5      21
6.0      51
6.5      81
7.0     140
7.5     112
8.0     123
8.5      72
9.0      60
9.5      16
10.0     14
Name: count, dtype: int64
score
low           54
mid_low      153
mid          252
high         195
very_high     90
Name: count, dtype: int64


In [48]:
#Stratified train-test split
X = clean_df["rev_text"]
y = clean_df["score"]

score_bins = pd.cut(
    y,
    bins=[-0.01, 5, 6.5, 7.5, 8.5, 10],
    labels=False
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=score_bins
)

print("Training:", X_train.shape)
print("Test:", X_test.shape)

Training: (595,)
Test: (149,)


In [49]:
#Distribution of sample sets check
print("Overall:")
print(score_bins.value_counts(normalize=True).sort_index())

print("\nTrain:")
print(pd.cut(
    y_train,
    bins=[-0.01, 5, 6.5, 7.5, 8.5, 10],
    labels=False
).value_counts(normalize=True).sort_index())

print("\nTest:")
print(pd.cut(
    y_test,
    bins=[-0.01, 5, 6.5, 7.5, 8.5, 10],
    labels=False
).value_counts(normalize=True).sort_index())

Overall:
score
0    0.072581
1    0.205645
2    0.338710
3    0.262097
4    0.120968
Name: proportion, dtype: float64

Train:
score
0    0.072269
1    0.205042
2    0.339496
3    0.262185
4    0.121008
Name: proportion, dtype: float64

Test:
score
0    0.073826
1    0.208054
2    0.335570
3    0.261745
4    0.120805
Name: proportion, dtype: float64


In [50]:


# Create strata for the TRAINING data only
train_score_bins = pd.cut(
    y_train,
    bins=[-0.01, 5, 6.5, 7.5, 8.5, 10],
    labels=False
)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)
#-------------
text_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("ridge", Ridge())
])
#===============
param_grid = {
    "tfidf__max_features": [5_000, 10_000, 20_000],
    "tfidf__ngram_range": [(1, 1), (1, 3), (1, 5)],
    "tfidf__min_df": [1, 2, 3],
    "ridge__alpha": [0.1, 1, 10, 100]
}

In [51]:
grid_search = GridSearchCV(
    text_pipeline,
    param_grid,
    cv=cv.split(X_train, train_score_bins),
    scoring="neg_mean_absolute_error",
    n_jobs=1,
    verbose=1
)

grid_search.fit(
    X_train,
    y_train
)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.","Pipeline(step...e', Ridge())])"
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'ridge__alpha': [0.1, 1, ...], 'tfidf__max_features': [5000, 10000, ...], 'tfidf__min_df': [1, 2, ...], 'tfidf__ngram_range': [(1, ...), (1, ...), ...]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'neg_mean_absolute_error'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",<generator ob...0015F63D909D0>
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` para

In [52]:
print("Best parameters:")
print(grid_search.best_params_)

print("\nBest CV MAE:")
print(-grid_search.best_score_)
#=====================
best_text_model = grid_search.best_estimator_

tuned_predictions = best_text_model.predict(X_test)

evaluate_model(
    "Tuned Text Model",
    tuned_predictions,
    y_test
)

Best parameters:
{'ridge__alpha': 0.1, 'tfidf__max_features': 20000, 'tfidf__min_df': 1, 'tfidf__ngram_range': (1, 5)}

Best CV MAE:
0.7447219058304355

Tuned Text Model
----------------
MAE:               0.822
RMSE:              1.164
R²:                0.427
Exact rating:      22.15%
Within 0.5 points: 59.73%


In [54]:
#NEXT: 
results = pd.DataFrame({
    "actual": y_test,
    "predicted": tuned_predictions,
    "error": tuned_predictions - y_test,
    "abs_error": abs(tuned_predictions - y_test),
    "text": X_test
})

results = results.sort_values("abs_error", ascending=False)

results.head(10)

,actual,predicted,error,abs_error,text
106,1.0,6.020111,5.020111,5.020111,glitterbug the wombats an mgmt wave band ive n...
517,2.5,6.991995,4.491995,4.491995,"the head hurts, but the heart knows the truth ..."
84,2.0,5.582544,3.582544,3.582544,<|°_°|> caravan palace oh god it's opening wit...
114,2.0,5.293949,3.293949,3.293949,reputation taylor swift hyperpop taylor?! im n...
330,3.0,6.053218,3.053218,3.053218,all the pretty horses current 93 experimental ...
276,2.0,4.782208,2.782208,2.782208,mother ship' dance gavin dance sounds like cra...
252,4.5,6.835204,2.335204,2.335204,songs for the deaf queens of the stone age ope...
147,10.0,7.732086,-2.267914,2.267914,the glow pt. 2 the microphones beautiful waves...
145,4.5,6.627376,2.127376,2.127376,why would i watch hot mulligan ok so imagine y...
10,5.5,7.459115,1.959115,1.959115,open season the high highs. airy piano opener....


In [63]:
new_review=r"poopoopeepee"

In [64]:

prediction = best_text_model.predict([new_review.lower().strip()])
print(prediction[0])

7.217401883192424
